<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/solutions/Session03_reviewing_and_testing_ai_code_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۳: بازبینی، دیباگ و تست کد تولیدشده با هوش مصنوعی

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session03_reviewing_and_testing_ai_code.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
import math
import os
from pathlib import Path

import pytest

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

تابع `parse_price` جلسه قبل را با تست‌های جدید محک بزنید: «۱٫۲۵۰٫۰۰۰ تومان»، رشته خالی، «٤٥ هزار تومان» با ارقام عربی و عددی با علامت جهت‌نمای نامرئی `\u200f`. اشکال‌ها را پیدا و رفع کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

نسخهٔ جلسهٔ قبل از `parse_price` را در فایل می‌نویسیم و تست‌های جدید را روی آن اجرا می‌کنیم:

</div>

In [ ]:
%%writefile price.py
import re

_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789")
_UNITS = {"میلیارد": 1_000_000_000, "میلیون": 1_000_000, "هزار": 1_000}


def parse_price(text: str) -> int:
    text = text.translate(_DIGITS).strip()
    if text == "رایگان":
        return 0
    text = text.replace("٫", ".").replace(",", "").replace("٬", "")
    match = re.search(r"\d+(?:\.\d+)?", text)
    if not match:
        raise ValueError(f"no number in {text!r}")
    value = float(match.group())
    for word, factor in _UNITS.items():
        if word in text:
            value *= factor
            break
    return int(round(value))

In [ ]:
%%writefile test_price.py
import pytest
from price import parse_price


@pytest.mark.parametrize("text, expected", [
    ("۴۵,۰۰۰ تومان", 45_000),
    ("۱۲٫۵ میلیون تومان", 12_500_000),
    ("۱٫۲۵۰٫۰۰۰ تومان", 1_250_000),
    ("٤٥ هزار تومان", 45_000),
    ("\u200f۴۵۰ هزار تومان", 450_000),
])
def test_valid(text, expected):
    assert parse_price(text) == expected


@pytest.mark.parametrize("text", ["", "  "])
def test_empty(text):
    with pytest.raises(ValueError):
        parse_price(text)

In [ ]:
!python -m pytest -q --tb=line test_price.py

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

اشکال‌ها:

- **«۱٫۲۵۰٫۰۰۰ تومان»:** هر دو «٫» به «.» تبدیل می‌شوند و الگو فقط `1.250` را می‌گیرد. نتیجه ۱ تومان می‌شود!
- **ارقام عربی (`٤٥`):** جدول تبدیل فقط ارقام فارسی را دارد. `float` ارقام عربی را می‌پذیرد، ولی الگوی `\d` هم آن‌ها را می‌پذیرد، پس این مورد ممکن است تصادفاً درست کار کند. با این حال تکیه بر این رفتار شکننده است و تبدیل صریح بهتر است.
- **رشتهٔ خالی:** خطای درستی می‌دهد، ولی پیام آن مبهم است.
- **علامت جهت‌نمای `\u200f`:** در این نسخه مشکلی ایجاد نمی‌کند، چون `re.search` از آن می‌گذرد. ولی اگر کسی بعداً برای مقایسه دقیق رشته از `text == "رایگان"` استفاده کند، این علامت نامرئی مقایسه را خراب می‌کند. حذف آن در ابتدای تابع امن‌تر است.

نسخهٔ اصلاح‌شده:

</div>

In [ ]:
%%writefile price.py
import re

_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")
_INVISIBLE = dict.fromkeys(map(ord, "\u200e\u200f\u202a\u202b\u202c\u202d\u202e\u2066\u2067\u2068\u2069\ufeff"))
_UNITS = {"میلیارد": 1_000_000_000, "میلیون": 1_000_000, "هزار": 1_000}
_GROUPED = re.compile(r"\d{1,3}(?:[.٫]\d{3})+(?![\d.٫])")


def parse_price(text: str) -> int:
    """Convert a Persian price string to an amount in toman."""
    text = text.translate(_INVISIBLE).translate(_DIGITS).strip()
    if not text:
        raise ValueError("empty price")
    if text == "رایگان":
        return 0
    text = text.replace(",", "").replace("٬", "")
    grouped = _GROUPED.search(text)
    if grouped:                                   # 1.250.000 -> 1250000
        number = grouped.group().replace(".", "").replace("٫", "")
    else:
        match = re.search(r"\d+(?:[.٫]\d+)?", text)
        if not match:
            raise ValueError(f"no number in {text!r}")
        number = match.group().replace("٫", ".")
    value = float(number)
    for word, factor in _UNITS.items():
        if word in text:
            value *= factor
            break
    return int(round(value))

In [ ]:
!python -m pytest -q --tb=line test_price.py

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

تابع `apply_discount` را طوری اصلاح کنید که تست `test_discount_rules` پاس شود و گرد کردن همیشه به سمت بالا باشد. تست مناسب برای گرد کردن هم بنویسید.

</div>

In [ ]:
def apply_discount(price: int, percent: float) -> int:
    """Apply a percentage discount and round UP to the next 1,000 toman."""
    if not 0 <= percent <= 100:
        raise ValueError(f"discount must be between 0 and 100, got {percent}")
    discounted = price * (100 - percent) / 100
    return int(math.ceil(discounted / 1000) * 1000)


def test_discount_rules():
    assert apply_discount(1_000_000, 0) == 1_000_000
    assert apply_discount(1_000_000, 100) == 0
    with pytest.raises(ValueError):
        apply_discount(1_000_000, 150)


def test_rounding_up():
    assert apply_discount(250_000, 15) == 213_000      # 212,500 -> 213,000
    assert apply_discount(1_001_000, 0) == 1_001_000   # already a multiple of 1,000
    assert apply_discount(999, 0) == 1_000             # always rounds up
    assert apply_discount(3_000_000, 33.3) == 2_001_000


test_discount_rules()
test_rounding_up()
print("all tests pass")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

`math.ceil` همیشه رو به بالا گرد می‌کند و مشکل گرد کردن بانکی `round` را ندارد. تست `test_rounding_up` چهار حالت را پوشش می‌دهد: مقدار وسط (۲۱۲٬۵۰۰)، مقداری که از قبل مضرب ۱۰۰۰ است، مقدار کوچک، و درصد اعشاری.

یک نکتهٔ ظریف: در محاسبات ممیز شناور، مقداری مثل `1_000_000 * 0.9` ممکن است کمی بیشتر از ۹۰۰٬۰۰۰ شود و `ceil` آن را به ۹۰۱٬۰۰۰ برساند. ما ابتدا در `100 - percent` ضرب و بعد بر ۱۰۰ تقسیم کردیم تا این خطا برای درصدهای صحیح پیش نیاید. برای حساب‌های مالی دقیق، از `decimal.Decimal` یا محاسبه با عدد صحیح (ریال) استفاده کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

از یک ابزار هوش مصنوعی بخواهید «تابعی بنویس که نام فایلی را از کاربر بگیرد و محتوایش را از پوشه `uploads` برگرداند». کد را از نظر امنیتی بررسی کنید. با ورودی `../../etc/passwd` چه می‌شود؟ (این حمله **Path Traversal** نام دارد.)

</div>

In [ ]:
os.makedirs("uploads", exist_ok=True)
Path("uploads/invoice.txt").write_text("فاکتور سفارش NK482913", encoding="utf-8")
Path("secret.txt").write_text("DB_PASSWORD=very-secret", encoding="utf-8")


# Typical AI-generated version
def read_upload_unsafe(filename):
    with open(os.path.join("uploads", filename), encoding="utf-8") as f:
        return f.read()


print(read_upload_unsafe("invoice.txt"))
print(read_upload_unsafe("../secret.txt"))        # escapes the uploads folder

In [ ]:
UPLOAD_DIR = Path("uploads").resolve()


def read_upload(filename):
    """Read a file from the uploads folder; refuse any path that leaves the folder."""
    path = (UPLOAD_DIR / filename).resolve()
    if not path.is_relative_to(UPLOAD_DIR) or not path.is_file():
        raise PermissionError(f"access denied: {filename!r}")
    return path.read_text(encoding="utf-8")


for name in ["invoice.txt", "../secret.txt", "../../etc/passwd", "/etc/passwd"]:
    try:
        print(f"{name!r}: {read_upload(name)}")
    except PermissionError as e:
        print(e)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

`os.path.join("uploads", "../../etc/passwd")` مسیری می‌سازد که از پوشهٔ `uploads` بیرون می‌رود، و اگر نام فایل مسیر مطلق باشد (`/etc/passwd`)، `join` بخش اول را کاملاً دور می‌ریزد. راه درست:

1. مسیر نهایی را با `resolve()` به شکل مطلق و بدون `..` تبدیل کنیم.
2. بررسی کنیم که هنوز داخل پوشهٔ مجاز است (`is_relative_to`).

فیلتر کردن رشتهٔ `..` با `replace` راه‌حل امنی نیست؛ مهاجم با `....//` یا کدگذاری‌های دیگر از آن عبور می‌کند. در وب، بهتر است اصلاً نام فایل را از کاربر نگیرید و به جای آن شناسه‌ای بگیرید که در دیتابیس به فایل نگاشت می‌شود.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

یک تابع ده‌خطی از پروژه‌های قبلی خودتان را به ابزار بدهید و تست بخواهید. چند تست از روی قواعد اضافه کنید که ابزار نوشته بود؟

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

این تمرین به کد خودتان بستگی دارد. نمونهٔ زیر همین روند را برای تابعی نشان می‌دهد که هزینهٔ ارسال را بر اساس قواعد نوین‌کالا حساب می‌کند. فرض کنید ابزار برای این تابع تست‌هایی نوشته است که همه از روی کد ساخته شده‌اند. تست‌های زیر از روی **سند سیاست ارسال** نوشته شده‌اند:

</div>

In [ ]:
FAST_CITIES = {"تهران", "کرج", "اصفهان", "شیراز", "مشهد"}


def shipping_cost(total, method, city):
    """Shipping cost in toman according to the Novin Kala shipping policy."""
    if method == "standard":
        return 0 if total > 1_000_000 else 45_000
    if method == "express":
        if city not in FAST_CITIES:
            raise ValueError(f"express delivery is not available in {city}")
        return 85_000
    raise ValueError(f"unknown method {method!r}")


def test_shipping_rules():
    assert shipping_cost(500_000, "standard", "آمل") == 45_000
    assert shipping_cost(1_000_001, "standard", "آمل") == 0
    assert shipping_cost(1_000_000, "standard", "آمل") == 45_000      # "more than one million"
    assert shipping_cost(5_000_000, "express", "تهران") == 85_000      # express is never free
    with pytest.raises(ValueError):
        shipping_cost(500_000, "express", "تبریز")


test_shipping_rules()
print("rule-based tests pass")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

تست‌های مبتنی بر قاعده معمولاً سه نوع مورد را اضافه می‌کنند که تست‌های تولیدشده از روی کد ندارند:

- **مرزها:** «بیش از یک میلیون» یعنی خود یک میلیون رایگان نیست. تست تولیدشده معمولاً مقدارهای دور از مرز را امتحان می‌کند.
- **استثناها:** «ارسال سریع شامل ارسال رایگان نمی‌شود». این قاعده در کد ممکن است فقط به‌طور ضمنی رعایت شده باشد.
- **ورودی‌های نامعتبر:** شهری که ارسال سریع ندارد، یا روش ارسال ناشناخته.

برای کد خودتان هم همین را بپرسید: کدام قاعدهٔ کسب‌وکار یا مسئله در تست‌ها نیامده است؟

</div>